## 4) Clasificador Spam / No Spam
Se construye un clasificador binario para el dataset `spam_ham_es_dataset_v1.csv`.
Se usa un embedding **TF‑IDF** (texto → vector numérico) y un modelo supervisado.
Se reportan métricas: Accuracy, Precision, Recall, F1 Score, ROC AUC, Binary Crossentropy.


In [7]:
import pandas as pd

csv_path = "spam_ham_es_dataset_v1.csv"
df = pd.read_csv(csv_path)

df["y"] = (df["tipo"].str.lower()=="spam").astype(int)
X = df["texto"].astype(str).values
y = df["y"].values

df.head()


,tipo,texto,y
0,spam,¡Felicidades! Ganaste un iPhone completamente ...,1
1,spam,Gana $500 al día trabajando desde casa. Regíst...,1
2,spam,Tu cuenta será suspendida. Verifica tu informa...,1
3,spam,Oferta limitada: consigue medicamentos con 70%...,1
4,spam,Has sido seleccionado para un premio exclusivo...,1


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

# Train / Validation / Test (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1,2), min_df=2)),
    ("clf", LogisticRegression(max_iter=2000, solver="liblinear", C=4))
])

model.fit(X_train, y_train)

val_prob = model.predict_proba(X_val)[:,1]
test_prob = model.predict_proba(X_test)[:,1]

val_pred = (val_prob>=0.5).astype(int)
test_pred = (test_prob>=0.5).astype(int)

def metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true,y_pred),
        "precision": precision_score(y_true,y_pred, zero_division=0),
        "recall": recall_score(y_true,y_pred, zero_division=0),
        "f1": f1_score(y_true,y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true,y_prob),
        "binary_crossentropy": log_loss(y_true, y_prob, labels=[0,1]),
    }

val_metrics = metrics(y_val, val_pred, val_prob)
test_metrics = metrics(y_test, test_pred, test_prob)

val_metrics, test_metrics


({'accuracy': 0.9422222222222222,
  'precision': 0.9304347826086956,
  'recall': 0.9553571428571429,
  'f1': 0.9427312775330396,
  'roc_auc': np.float64(0.9866466498103666),
  'binary_crossentropy': 0.20014170295954792},
 {'accuracy': 0.9511111111111111,
  'precision': 0.954954954954955,
  'recall': 0.9464285714285714,
  'f1': 0.9506726457399103,
  'roc_auc': np.float64(0.9901232616940582),
  'binary_crossentropy': 0.21340722538924647})